# End-to-end demo — pipeline SAM3-only sobre un video real completo

Escenario **real, video entero** (sin cuotas, sin clips), pensado para correr **en el pod (GPU)**.

- Selecciona **al azar** un video del split de **reserva** (`split = 0`).
- Carga **SAM3 una sola vez** y lo reutiliza.
- **Prueba A — segmentación** (video entero, `include_masks=True`, con mp4).
- **Prueba B — tracking** (video entero, `include_masks=True`, con mp4) + **overlay por `obj_id`**.
- Celdas para **visualizar** el video original, el segmentado y el de tracking.

> ⚠️ **Costo:** procesar un video completo con SAM3 por frame y por clase (+ máscaras + render) puede tardar **bastante** (minutos a decenas de minutos según duración/fps). Es intencional: es la demo de punta a punta sobre material real.

## Setup

In [1]:
import random
from pathlib import Path

import pandas as pd
from IPython.display import Video, display

from src.utils import PROJECT_ROOT, get_abs_path
from src.data.metadata import _load_metadata_config
from src.core.sam3_loader import load_sam3
from src.core.inference import run_inference
from src.core.track_overlay import render_obj_id_overlay
from src.core.frame_extraction import iter_frames, get_video_fps
from src.core.video_writer import open_video_writer

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /workspace/FutBotMX-UAQTeam


## 1. Seleccionar un video de reserva al azar

`SEED` fija la selección (reproducible). Pon `SEED = None` para azar puro en cada ejecución.

In [2]:
SEED = 0

_, metadata_csv, _, _ = _load_metadata_config()
df = pd.read_csv(get_abs_path(metadata_csv))
reserved = df[df["split"] == 0].sort_values("id")
if reserved.empty:
    raise RuntimeError("No hay videos en el split de reserva (split=0).")

rng = random.Random(SEED)
row = reserved.iloc[rng.randrange(len(reserved))]
VIDEO = row["ruta"]
print(f"Video elegido (reserva): id={row['id']}  {VIDEO}")
print(
    f"  duracion={row['duracion']:.1f}s  {row['ancho']}x{row['alto']}"
    f"  fps≈{row['fps_average']:.1f}"
)

Video elegido (reserva): id=74  data/raw/17Abril/video-615_singular_display.mov
  duracion=34.1s  1328x1760  fps≈59.7


## 2. Cargar SAM3 una sola vez

In [3]:
bundle = load_sam3()
print("SAM3 cargado:", bundle.device)

Loading weights:   0%|          | 0/1797 [00:00<?, ?it/s]

SAM3 cargado: cuda


## 3. Prueba A — Segmentación (video entero, con máscaras)

`sampling="all"` → **todos** los frames (sin cuota). `include_masks=True` → máscaras COCO-RLE en el JSON. `render_video=True` → mp4 anotado.

In [4]:
res_a = run_inference(
    VIDEO,
    mode="segmentation",
    sampling="all",        # video entero (sin cuota, sin clip)
    include_masks=True,    # máscaras COCO-RLE en el JSON
    render_video=True,
    bundle=bundle,
)
print("A (segmentación):")
print("  json :", res_a["json"])
print("  video:", res_a["video"])

  frame 1/2037


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

  frame 2/2037
  frame 3/2037
  frame 4/2037
  frame 5/2037
  frame 6/2037
  frame 7/2037
  frame 8/2037
  frame 9/2037
  frame 10/2037
  frame 11/2037
  frame 12/2037
  frame 13/2037
  frame 14/2037
  frame 15/2037
  frame 16/2037
  frame 17/2037
  frame 18/2037
  frame 19/2037
  frame 20/2037
  frame 21/2037
  frame 22/2037
  frame 23/2037
  frame 24/2037
  frame 25/2037
  frame 26/2037
  frame 27/2037
  frame 28/2037
  frame 29/2037
  frame 30/2037
  frame 31/2037
  frame 32/2037
  frame 33/2037
  frame 34/2037
  frame 35/2037
  frame 36/2037
  frame 37/2037
  frame 38/2037
  frame 39/2037
  frame 40/2037
  frame 41/2037
  frame 42/2037
  frame 43/2037
  frame 44/2037
  frame 45/2037
  frame 46/2037
  frame 47/2037
  frame 48/2037
  frame 49/2037
  frame 50/2037
  frame 51/2037
  frame 52/2037
  frame 53/2037
  frame 54/2037
  frame 55/2037
  frame 56/2037
  frame 57/2037
  frame 58/2037
  frame 59/2037
  frame 60/2037
  frame 61/2037
  frame 62/2037
  frame 63/2037
  frame 64/2037


## 4. Prueba B — Tracking (video entero, con máscaras)

`sampling="auto"` + `max_frames=None` → tracking sobre el **video completo** (prefijo contiguo entero, requisito de ByteTrack).

In [5]:
res_b = run_inference(
    VIDEO,
    mode="tracking",
    sampling="auto",       # contiguo completo
    max_frames=None,       # video entero
    include_masks=True,
    render_video=True,
    bundle=bundle,
)
print("B (tracking):")
print("  json :", res_b["json"])
print("  video:", res_b["video"])
print("  tracks:", len(res_b["index"]), "obj_id únicos")

B (tracking):
  json : /workspace/FutBotMX-UAQTeam/outputs/inference/video-615_singular_display/video-615_singular_display.json
  video: /workspace/FutBotMX-UAQTeam/outputs/inference/video-615_singular_display/video-615_singular_display.mp4
  tracks: 18 obj_id únicos


## 5. Overlay por `obj_id` (hace visible el tracking)

Post-pase sobre el JSON de B: caja + etiqueta `nombre #id` + trayectoria por objeto (color por clase), excluyendo `green_floor`. Como el JSON trae `rle`, activamos también el relleno de máscara.

In [6]:
overlay_mp4 = render_obj_id_overlay(
    res_b["json"],
    draw_masks=True,   # máscara + caja + etiqueta + trayectoria
)
print("Overlay obj_id:", overlay_mp4)

Overlay obj_id: /workspace/FutBotMX-UAQTeam/outputs/inference/video-615_singular_display/video-615_singular_display_obj_id.mp4


## 6. Visualización

Corre cada celda para ver el video correspondiente **inline**. Si un video no se reproduce en tu navegador, pon `embed=True` en `show_video` (lo incrusta en base64; pesado para videos largos).

In [7]:
def show_video(path, width=720, embed=False):
    """Muestra un mp4 inline."""
    display(Video(str(path), width=width, embed=embed))


def to_browser_mp4(video, dst=None):
    """Re-encoda un video (p. ej. .MOV) a mp4 navegable, a su fps real, video entero."""
    video = Path(video)
    stem = video.stem
    if dst is None:
        dst = PROJECT_ROOT / "outputs" / "inference" / stem / f"{stem}_orig.mp4"
    fps = get_video_fps(video)
    with open_video_writer(dst, fps=fps) as append:
        for _, frame in iter_frames(video):
            append(frame)
    return dst

### Video original

El dataset es `.MOV`; se re-encoda a mp4 navegable (video entero) para verlo inline.

In [8]:
orig_mp4 = to_browser_mp4(VIDEO)
show_video(orig_mp4)

### Segmentación (Prueba A)

In [9]:
show_video(res_a["video"])

### Tracking con overlay por `obj_id` (Prueba B)

In [10]:
show_video(overlay_mp4)

### (Opcional) mp4 de tracking "crudo" (overlay por clase, sin identidad)

El que produce `run_inference` en B: útil para comparar contra el overlay por `obj_id`.

In [11]:
show_video(res_b["video"])